# 🔮 FORESIGHT — 10: MLOps, Experiment Tracking & Drift Monitoring

This notebook demonstrates:
- **Population Stability Index (PSI) Distribution Shift Calculations**
- **Kolmogorov-Smirnov (KS) Non-Parametric Divergence Tests**
- **Concept Drift Tracking & Rolling WAPE Degradation**
- **Automated MLflow Experiment Tracking & Governance**

In [ ]:
import numpy as np
import pandas as pd
from foresight.mlops.drift_detector import calculate_psi, calculate_ks_test, DriftDetector

# 1. PSI Demonstration: Stable vs Drifted Distributions
np.random.seed(42)
expected = np.random.normal(loc=50, scale=10, size=5000)
actual_stable = np.random.normal(loc=50.2, scale=9.8, size=2000)
actual_drifted = np.random.normal(loc=62.0, scale=14.0, size=2000)

psi_stable = calculate_psi(expected, actual_stable)
psi_drift = calculate_psi(expected, actual_drifted)

print("=== POPULATION STABILITY INDEX (PSI) BENCHMARKS ===")
print(f"Stable Production Distribution:  PSI = {psi_stable:.4f} (Threshold < 0.10 -> NO DRIFT)")
print(f"Drifted Production Distribution: PSI = {psi_drift:.4f} (Threshold > 0.20 -> SIGNIFICANT DRIFT)")

## 2. Concept Drift & Automated Retraining Decision Matrix

In [ ]:
detector = DriftDetector()

# Simulate 3 production scenarios
y_true = np.random.uniform(10, 50, 1000)
y_pred_healthy = y_true + np.random.normal(0, 4, 1000)
y_pred_degraded = y_true * 0.7 + np.random.normal(0, 12, 1000)

healthy_res = detector.evaluate_concept_drift(y_true, y_pred_healthy, baseline_wape=0.1894)
degraded_res = detector.evaluate_concept_drift(y_true, y_pred_degraded, baseline_wape=0.1894)

print("=== HEALTHY PRODUCTION EVALUATION ===")
print(f"Rolling WAPE: {healthy_res.current_rolling_wape:.2%} | Degradation: {healthy_res.wape_degradation_pct:+.1f}%")
print(f"Action: {healthy_res.retraining_action.value} -> {healthy_res.justification}")

print("\n=== DEGRADED PRODUCTION EVALUATION ===")
print(f"Rolling WAPE: {degraded_res.current_rolling_wape:.2%} | Degradation: {degraded_res.wape_degradation_pct:+.1f}%")
print(f"Action: {degraded_res.retraining_action.value} -> {degraded_res.justification}")